# Prepare Gender and Age-Group Public Benchmark Data

This notebook materializes labelled internal train/val/test splits for the public `gender` and `age_group` datasets. official hidden/public test files are ignored to avoid leakage.


In [ ]:
# Cell 1 - Setup & Install
import json
import os
import pickle
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import yaml
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyyaml", "-q"])
    import yaml

REPO_DIR = Path("/kaggle/working/denoising-event-sequences")
if not REPO_DIR.exists():
    REPO_DIR = Path.cwd()
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DATA_ROOT = Path("/kaggle/working/data")
RAW_ROOT = DATA_ROOT / "raw"
PROCESSED_ROOT = DATA_ROOT / "processed"
OUTPUT_ROOT = Path("/kaggle/working/outputs/public_benchmarks")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(cmd, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w") as log_file:
        proc = subprocess.Popen(
            [str(x) for x in cmd],
            cwd=str(REPO_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")


In [ ]:
# Cell 2 - Dataset Selection
DATASETS_TO_PREPARE = ["gender", "age_group"]  # or ["gender"] / ["age_group"]
FORCE_REBUILD_PROCESSED = False
MAX_SEQ_LEN = 512
MAX_ENTITIES = None  # set a small integer for a quick smoke run

for dataset_name in DATASETS_TO_PREPARE:
    assert dataset_name in {"gender", "age_group"}
print(DATASETS_TO_PREPARE)


In [ ]:
# Cell 3 - Run Preparation Script
reports = []
for dataset_name in DATASETS_TO_PREPARE:
    processed_dir = PROCESSED_ROOT / f"{dataset_name}_benchmark"
    if FORCE_REBUILD_PROCESSED and processed_dir.exists():
        shutil.rmtree(processed_dir)
    cmd = [
        sys.executable,
        REPO_DIR / "scripts" / "prepare_public_benchmark_data.py",
        "--dataset", dataset_name,
        "--raw-root", RAW_ROOT,
        "--output-dir", processed_dir,
        "--max-seq-len", MAX_SEQ_LEN,
    ]
    if MAX_ENTITIES is not None:
        cmd += ["--max-entities", MAX_ENTITIES]
    run_logged(cmd, OUTPUT_ROOT / "logs" / f"{dataset_name}_prepare_public_benchmark_data.log")
    with (processed_dir / "data_report.json").open() as f:
        reports.append(json.load(f))

reports_df = pd.DataFrame(reports)
reports_df


In [ ]:
# Cell 4 - Inspect Artifacts and Split Balance
summary_rows = []
for dataset_name in DATASETS_TO_PREPARE:
    processed_dir = PROCESSED_ROOT / f"{dataset_name}_benchmark"
    events = pd.read_parquet(processed_dir / "events.parquet")
    canonical = pd.read_parquet(processed_dir / "canonical_events.parquet")
    with (processed_dir / "splits.json").open() as f:
        splits = json.load(f)
    with (processed_dir / "prepared_config.yaml").open() as f:
        cfg = yaml.safe_load(f)
    entity_col = cfg["data"]["group_col"]
    target_col = cfg["data"]["target_col"]
    for split_name, ids in splits.items():
        labels = canonical[canonical[entity_col].isin(ids)].groupby(entity_col)[target_col].first()
        summary_rows.append({
            "dataset": dataset_name,
            "split": split_name,
            "entities": len(ids),
            "rows": int(canonical[canonical[entity_col].isin(ids)].shape[0]),
            "class_counts": dict(labels.value_counts().sort_index()),
        })
    print(dataset_name, events.shape, canonical.shape)

split_summary_df = pd.DataFrame(summary_rows)
split_summary_df


In [ ]:
# Cell 5 - Artifact Summary
for dataset_name in DATASETS_TO_PREPARE:
    processed_dir = PROCESSED_ROOT / f"{dataset_name}_benchmark"
    print(f"[{dataset_name}] {processed_dir}")
    for name in ["canonical_events.parquet", "events.parquet", "transformed_events.parquet", "splits.json", "preprocessor.pkl", "prepared_config.yaml", "data_report.json"]:
        print(" ", name, "OK" if (processed_dir / name).exists() else "MISSING")
